# Bootstrap CI for MEL recall / precision (Correction 1, part 1)


No retraining, no GPU needed. Uses your existing `test_probs_seed*.csv` (fixed-weight) and the equivalent adaptive-weight CSVs. Edit `FIXED_PATTERN` / `ADAPTIVE_PATTERN` below, then run all cells.

In [2]:
# ============================================================
# Bootstrap confidence intervals for MEL recall / precision /
# macro-F1 / accuracy, computed from EXISTING saved per-seed
# test-set probability CSVs. No retraining, no GPU needed.
#
# DATASETS NEEDED (edit the two glob patterns below):
#   - Fixed-weight test probabilities: your existing
#     test_probs_seed{seed}.csv files (from the SSL+rebalance
#     fixed-weight fine-tuning notebook). Columns expected:
#     BKL, MEL, NV, true_label (same format as val_probs_seed*.csv).
#   - Adaptive-weight test probabilities: the equivalent CSVs
#     saved by the adaptive-weighting fine-tuning notebook.
#     If you named them differently, edit ADAPTIVE_PATTERN.
#
# OUTPUT:
#   - bootstrap_ci_results.csv: one row per (model, seed, metric)
#     with point estimate, 2.5th and 97.5th percentile (95% CI).
#   - A printed summary table pooling all seeds together per
#     model, for direct quoting in the paper (e.g. "MEL recall =
#     41.7% [95% CI: X-Y%]" instead of just mean +/- SD).
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# ---- EDIT THESE TWO PATTERNS ----
FIXED_PATTERN    = '/kaggle/input/datasets/ittisamurtunib/path-3seed-ssl-finetune/test_probs_seed{seed}.csv'
ADAPTIVE_PATTERN = '/kaggle/input/datasets/ittisamurtunib/val-probs-ssl-rebalance/val_probs_seed{seed}.csv'
# ----------------------------------

OUTPUT_DIR = '/kaggle/working'
SEEDS = [42, 123, 2024]
N_BOOT = 2000
RNG_SEED = 0
CLASSES = ['BKL', 'MEL', 'NV']


def load_probs(pattern, seed):
    df = pd.read_csv(pattern.format(seed=seed))
    assert set(CLASSES + ['true_label']).issubset(df.columns), f"Unexpected columns: {df.columns.tolist()}"
    return df


def metrics_from_df(df):
    preds = df[CLASSES].values.argmax(axis=1)
    pred_labels = np.array([CLASSES[i] for i in preds])
    true_labels = df['true_label'].values
    acc = np.mean(pred_labels == true_labels)
    macro_f1 = f1_score(true_labels, pred_labels, labels=CLASSES, average='macro', zero_division=0)
    mel_true = true_labels == 'MEL'
    mel_pred = pred_labels == 'MEL'
    tp = np.sum(mel_true & mel_pred)
    fn = np.sum(mel_true & ~mel_pred)
    fp = np.sum(~mel_true & mel_pred)
    mel_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    mel_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return {'accuracy': acc, 'macro_f1': macro_f1, 'mel_recall': mel_recall, 'mel_precision': mel_precision}


def bootstrap_metrics(df, n_boot, rng):
    n = len(df)
    rows = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        rows.append(metrics_from_df(df.iloc[idx]))
    return pd.DataFrame(rows)


def summarize(boot_df, point_metrics, model, seed):
    out = []
    for metric in ['accuracy', 'macro_f1', 'mel_recall', 'mel_precision']:
        lo, hi = np.percentile(boot_df[metric], [2.5, 97.5])
        out.append({'model': model, 'seed': seed, 'metric': metric,
                    'point_estimate': point_metrics[metric], 'ci_low': lo, 'ci_high': hi})
    return out


rng = np.random.default_rng(RNG_SEED)
all_rows = []
pooled = {'fixed': [], 'adaptive': []}

for model_name, pattern in [('fixed', FIXED_PATTERN), ('adaptive', ADAPTIVE_PATTERN)]:
    for seed in SEEDS:
        df = load_probs(pattern, seed)
        point = metrics_from_df(df)
        boot = bootstrap_metrics(df, N_BOOT, rng)
        all_rows.extend(summarize(boot, point, model_name, seed))
        pooled[model_name].append(df)

results_df = pd.DataFrame(all_rows)
out_path = f'{OUTPUT_DIR}/bootstrap_ci_results.csv'
results_df.to_csv(out_path, index=False)
print(f"Wrote per-seed CIs -> {out_path}\n")
print(results_df.to_string(index=False))

print("\n--- Pooled across 3 seeds (all 312 test predictions per model) ---")
print("Use this pooled number as your headline CI in the paper.\n")
for model_name in ['fixed', 'adaptive']:
    pooled_df = pd.concat(pooled[model_name], ignore_index=True)
    point = metrics_from_df(pooled_df)
    boot = bootstrap_metrics(pooled_df, N_BOOT, rng)
    print(f"{model_name.upper()}-WEIGHT MODEL (n={len(pooled_df)} pooled predictions):")
    for metric in ['accuracy', 'macro_f1', 'mel_recall', 'mel_precision']:
        lo, hi = np.percentile(boot[metric], [2.5, 97.5])
        print(f"  {metric:15s}: {point[metric]:.3f}  [95% CI: {lo:.3f}, {hi:.3f}]")
    print()


Wrote per-seed CIs -> /kaggle/working/bootstrap_ci_results.csv

   model  seed        metric  point_estimate   ci_low  ci_high
   fixed    42      accuracy        0.625000 0.528846 0.721154
   fixed    42      macro_f1        0.496814 0.388382 0.605816
   fixed    42    mel_recall        0.375000 0.000000 0.750000
   fixed    42 mel_precision        0.200000 0.000000 0.421053
   fixed   123      accuracy        0.625000 0.538462 0.721154
   fixed   123      macro_f1        0.493834 0.386204 0.599192
   fixed   123    mel_recall        0.500000 0.142411 0.875000
   fixed   123 mel_precision        0.190476 0.045455 0.384615
   fixed  2024      accuracy        0.692308 0.605769 0.778846
   fixed  2024      macro_f1        0.540333 0.427486 0.657693
   fixed  2024    mel_recall        0.375000 0.000000 0.750000
   fixed  2024 mel_precision        0.214286 0.000000 0.500000
adaptive    42      accuracy        0.644231 0.557692 0.730769
adaptive    42      macro_f1        0.477884 0.363558 